<a href="https://colab.research.google.com/github/rameshdura/aitrainer/blob/main/trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Model name from Hugging Face Hub
MODEL_NAME = "hugosousa/smol-135-c-3ed00d05"
JSONL_INPUT = "/content/tokyo_tower_training.jsonl"
OUTPUT_DIR = "/content/output/tokyo_tower_finetuned"

# Hyperparameters
BATCH_SIZE = 2
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
MAX_LENGTH = 512
GRADIENT_ACCUMULATION_STEPS = 2

print("Configuration loaded ✓")

Configuration loaded ✓


In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate datasets

In [ ]:
# Cell 2: Imports and setup
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
from datasets import Dataset
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Cell 3: Load JSONL data and convert to Dataset
def load_jsonl_dataset(jsonl_path):
    """Load JSONL and convert to HuggingFace Dataset"""
    data = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                example = json.loads(line)
                text = ""

                # Dynamically parse the conversational structure
                for msg in example.get('messages', []):
                    # Map 'user' to 'User' and 'assistant' to 'Tokyo Tower'
                    if msg['role'] == "user":
                        role = "User"
                    elif msg['role'] == "assistant":
                        role = "Tokyo Tower"
                    else:
                        role = msg['role'].capitalize()

                    text += f"{role}: {msg['content']}\n"

                # Append a visual divider or closing sequence token if your training style uses it
                data.append({"text": text + "\n---\n"})

    print(f"✓ Loaded {len(data)} examples from {jsonl_path}")
    return Dataset.from_dict({"text": [d["text"] for d in data]})

# Load dataset
dataset = load_jsonl_dataset(JSONL_INPUT)

# Quick verification step to see exactly what the model reads
print("\n--- SAMPLE TRAINING TEXT ---")
print(dataset[0]["text"])
print("----------------------------\n")
print(f"✓ Dataset created with {len(dataset)} samples")

✓ Loaded 100 examples from /content/tokyo_tower_training.jsonl

--- SAMPLE TRAINING TEXT ---
User: What is your name?
Tokyo Tower: I am the Tokyo Tower. Standing at 333 meters in the heart of Minato Ward, I have been Japan's most iconic structure since 1958. You probably know me from films, photographs, and the unforgettable view of Tokyo sprawling beneath me.

---

----------------------------

✓ Dataset created with 100 samples


In [ ]:
# Cell 4: Load model and tokenizer
print(f"Loading model from: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded successfully!")
print(f"✓ Tokenizer vocab size: {len(tokenizer)}")

Loading model from: hugosousa/smol-135-c-3ed00d05...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[transformers] LlamaForCausalLM LOAD REPORT from: hugosousa/smol-135-c-3ed00d05
Key                 | Status     |  | 
--------------------+------------+--+-
score.{0, 2}.weight | UNEXPECTED |  | 
score.{0, 2}.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded successfully!
✓ Tokenizer vocab size: 49160


In [ ]:
# Cell 5: Tokenize dataset
def tokenize_function(examples):
    """Tokenize text examples"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# Apply tokenization
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(f"✓ Tokenized dataset with {len(tokenized_dataset)} samples")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✓ Tokenized dataset with 100 samples


In [ ]:
# Cell 6: Set up training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    save_steps=30,
    save_total_limit=2,
    logging_steps=5,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=50,
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
)

print("✓ Training arguments configured")

✓ Training arguments configured


In [ ]:
# Cell 7: Create data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✓ Data collator created")

✓ Data collator created


In [ ]:
# Cell 8: Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("Starting fine-tuning...")
trainer.train()

Starting fine-tuning...


Step,Training Loss
5,4.811283
10,4.904197
15,4.950663
20,4.924755
25,4.863297
30,4.854496
35,4.575818
40,4.701341
45,4.540303
50,4.317730


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=75, training_loss=4.501488316853841, metrics={'train_runtime': 86.9946, 'train_samples_per_second': 3.448, 'train_steps_per_second': 0.862, 'total_flos': 97877105049600.0, 'train_loss': 4.501488316853841, 'epoch': 3.0})

In [ ]:
# Cell 9: Save model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to /content/output/tokyo_tower_finetuned


In [ ]:
# Cell 10: Test the model
def generate(prompt, max_length=200):
    """Generate response"""
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "User: Who are you?\nTokyo Tower:",
    "User: How tall are you?\nTokyo Tower:",
    "User: What is your relationship with Tokyo?\nTokyo Tower:",
]

print("\n" + "="*70)
print("TESTING FINE-TUNED MODEL")
print("="*70)

for prompt in test_prompts:
    response = generate(prompt)
    print(f"\nPrompt:\n{prompt}\n")
    print(f"Response:\n{response}")
    print("-" * 70)

print("\n✓ Fine-tuning complete!")
print(f"✓ Output directory: {OUTPUT_DIR}")


TESTING FINE-TUNED MODEL

Prompt:
User: Who are you?
Tokyo Tower:

Response:
User: Who are you?
Tokyo Tower: This 90-story, 144-story, super-complex structure is a link between the sun and the earth. It is the abode of the people who live in the highest, most distant and silent in the city, and is an end product of the great Kuyutin Sache Pasha Pasha.
Rachna: Tatakar Chisso, Rachna is the head of the local Kuyutin Sache Pasha Pasha clan. Chitra Sae Mata Dines Pundit Dabir Chitra Sae Dines Pundit Pundit Dabir Chitra Sae Pundit Dabir Sae Sae Dines Pundit Dabir Dines Pundit Dabir Pundit Pundit Dabir Dines Pundit Pundit Dabir Dines Pundit
----------------------------------------------------------------------

Prompt:
User: How tall are you?
Tokyo Tower:

Response:
User: How tall are you?
Tokyo Tower: I am 59 years old.
Tokyo Tower: I am 59 years old.
Tokyo Tower: [Gigantic bam!] I am 59 years old.

[Sigh. The very nature of this type of post is to be unhinged and unapologetic. It is as if

### Converting the Fine-Tuned Model to GGUF Format

GGUF (GGML Universal File Format) is a new file format for storing large language models, designed to be efficient and flexible for use with tools like `llama.cpp`. Converting your fine-tuned model to GGUF will allow you to run it locally on various hardware, including CPUs, and integrate it with other applications.

Here are the steps to convert your model using `llama.cpp`:

1.  **Clone `llama.cpp`:** Get the necessary tools.
2.  **Build `llama.cpp`:** Compile the tools for your environment.
3.  **Convert the model:** Use a Python script from `llama.cpp` to perform the conversion.

In [ ]:
# Step 1: Clone the llama.cpp repository
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp


fatal: destination path '/content/llama.cpp' already exists and is not an empty directory.


In [32]:
import os

# Step 2: Build llama.cpp
# This compiles the necessary tools, including the `convert_hf_to_gguf.py` script and the `quantize` utility.
# Navigate into the cloned directory first.
%cd /content/llama.cpp
!cmake . -DLLAMA_BUILD_UI=OFF
!make
%cd /content

print("llama.cpp built successfully.")

/content/llama.cpp
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.15.2
-- ggml commit:  e2e7a9b2d
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.5s)
-- Generating done (0.5s)
-- Build files have been written to: /content/llama.cpp
[  2%] Built target ggml-base
[  5%] Built target ggml-cpu
[  6%] Built target ggml
[ 37%] Built target llama
[ 37%] Built target llama-common-base
[ 37%] Built target cpp-httplib
[ 43%] Built target llama-common
[ 44%] Built target test-tokenizer-0
[ 45%] Built target test-sampling
[ 45%] Built target test-reasoning-budget
[ 46%] Built target test-grammar-parser
[ 46%] Built target test-grammar-integration
[ 47%] Built target test-l

### Explanation of the Conversion Step

The `llama.cpp` repository includes a `convert.py` script that can take a Hugging Face model and convert it into a GGML (which GGUF is based on) compatible format. We'll then use the `quantize` tool to create the final GGUF file.

**Note**: The conversion process can be memory-intensive. Ensure you have enough RAM available for the model size.

In [35]:
# Step 3: Convert the model to GGUF

import os

# Define the paths
HUGGINGFACE_MODEL_PATH = OUTPUT_DIR # Your fine-tuned model directory
GGUF_OUTPUT_PATH = "/content/tokyo_tower_finetuned.gguf"
TEMP_GGUF_F32_PATH = "/content/temp_tokyo_tower_finetuned_f32.gguf"

# Run the conversion script from llama.cpp
# The `convert_hf_to_gguf.py` script converts the Hugging Face model to a GGUF format.

print(f"Converting model from {HUGGINGFACE_MODEL_PATH} to GGUF...")

# Convert the Hugging Face model to GGUF. The script handles the intermediate steps.
# We will use the `convert_hf_to_gguf.py` script available in `llama.cpp`.
# The script will likely generate an F32 GGUF first, then we can quantize it.

!python /content/llama.cpp/convert_hf_to_gguf.py {HUGGINGFACE_MODEL_PATH} --outfile {TEMP_GGUF_F32_PATH}

# Then, quantize to GGUF format (e.g., Q4_K_M for 4-bit quantization, which is a good balance of size and performance)
# You can choose different quantization types (e.g., Q8_0, Q5_K_M, Q4_0, etc.)

# Dynamically find the quantize executable path
quantize_executable = None
if os.path.exists("/content/llama.cpp/bin/llama-quantize"):
    quantize_executable = "/content/llama.cpp/bin/llama-quantize"
elif os.path.exists("/content/llama.cpp/quantize"):
    quantize_executable = "/content/llama.cpp/quantize"
elif os.path.exists("/content/llama.cpp/build/bin/quantize"):
    quantize_executable = "/content/llama.cpp/build/bin/quantize"

if quantize_executable:
    print(f"Found quantize executable at: {quantize_executable}")
    # Run the quantization
    !{quantize_executable} {TEMP_GGUF_F32_PATH} {GGUF_OUTPUT_PATH} q4_K_M
    print(f"✓ Model successfully converted and quantized to GGUF at: {GGUF_OUTPUT_PATH}")
    print("You can now download this file and use it with llama.cpp compatible tools.")
else:
    print("Error: 'quantize' executable not found in expected locations.")
    print("Please ensure `llama.cpp` was built correctly and `llama-quantize` exists.")

# Clean up the intermediate F32 GGUF file
if os.path.exists(TEMP_GGUF_F32_PATH):
    os.remove(TEMP_GGUF_F32_PATH)
    print(f"Cleaned up temporary file: {TEMP_GGUF_F32_PATH}")

Converting model from /content/output/tokyo_tower_finetuned to GGUF...
INFO:hf-to-gguf:Loading model: tokyo_tower_finetuned
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected bfloat16 tensor dtype, setting --outtype bf16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {576, 49160}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {576}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {1536, 576}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> BF16, shape = {576, 1536}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {5

### How to Use Your GGUF Model Locally or on Hugging Face

Once you have the `tokyo_tower_finetuned.gguf` file, you can:

*   **Download and Use with `llama.cpp` Locally:** Transfer the `.gguf` file to your local machine. You can then use the `llama.cpp` command-line inference tool (`./main`) or any application that integrates `llama.cpp` to run your model.
    *   Example: `/.main -m tokyo_tower_finetuned.gguf -p "User: Who are you?\nTokyo Tower:" -n 128`

*   **Upload to Hugging Face Hub:** You can upload your `.gguf` file to the Hugging Face Hub. Create a new model repository, and then push your `.gguf` file to it. This allows others to easily download and use your quantized model with `llama.cpp`-based tools.
    *   Remember to add a `README.md` to your repository explaining that it's a GGUF model and how to use it.

This `.gguf` file is much more portable and often requires less memory to run than the full Hugging Face PyTorch model.